# ⚖️ LLM-as-Judge Prompt Selection via RL Multi-Armed Bandits

This notebook demonstrates how to use **Multi-Armed Bandit** algorithms to select the
best **LLM-as-Judge prompt template** for evaluating AI-generated responses.

## Problem Statement

When using an LLM to judge the quality of another LLM's output, the **judge prompt**
matters enormously. Different prompt templates may:
- Score with different granularity and calibration
- Agree more or less with human expert ratings
- Produce different levels of consistency across runs
- Vary in cost (token usage) and latency

We frame this as a **multi-armed bandit** problem:

```
Arms:    [judge_prompt_A, judge_prompt_B, ..., judge_prompt_K]
Reward:  Agreement with human reference labels
Goal:    Discover the prompt that best agrees with human ratings
```

## Architecture

```
┌──────────────────────────────────────────────────────────────────────────────┐
│             LLM-AS-JUDGE PROMPT SELECTION (Real LLM Calls)                  │
├──────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  ┌─────────┐    ┌──────────────────┐    ┌──────────────────┐                │
│  │ EVAL    │───►│  BANDIT AGENT   │───►│  JUDGE PROMPT    │                │
│  │ DATASET │    │  (RL Selector)  │    │  POOL            │                │
│  └─────────┘    │                  │    │                  │                │
│       │         │ • Thompson       │    │ • Structured     │                │
│  Human │        │   Sampling       │    │ • Rubric-based   │                │
│  Refs  │        │ • UCB            │    │ • CoT-first      │                │
│       │         │ • Epsilon-Greedy  │    │ • Reference-     │                │
│       │         └──────────────────┘    │   anchored       │                │
│       │                  │              │ • Checklist       │                │
│       │                  ▼              │ • Strict critic   │                │
│       │         ┌──────────────────┐    └──────────────────┘                │
│       └────────►│  REWARD SIGNAL  │                                        │
│                 │                  │                                        │
│                 │ Agreement with   │                                        │
│                 │ human reference  │                                        │
│                 │ labels           │                                        │
│                 └──────────────────┘                                        │
└──────────────────────────────────────────────────────────────────────────────┘
```

## What You Will Learn

1. How different judge prompt designs affect evaluation quality
2. How Thompson Sampling, UCB, and Epsilon-Greedy compare at finding the best judge prompt
3. How to measure agreement between LLM judges and human reference labels
4. How to use LangGraph to orchestrate the bandit evaluation loop
5. How to build a production-ready adaptive judge selector

## Prerequisites

- `OPENAI_API_KEY` environment variable set
- `ANTHROPIC_API_KEY` environment variable set

## 1. Environment Setup

In [ ]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn \
    langchain langchain-openai langchain-anthropic langgraph \
    langchain-core pydantic tiktoken

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional, Any, TypedDict
from dataclasses import dataclass, field
from enum import Enum
from abc import ABC, abstractmethod
from collections import defaultdict
import random
import warnings
warnings.filterwarnings('ignore')

# LangChain imports
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

# LangGraph imports
from langgraph.graph import StateGraph, END, START

np.random.seed(42)
random.seed(42)

print("✅ All imports successful!")

In [ ]:
assert os.environ.get("OPENAI_API_KEY"), "❌ OPENAI_API_KEY not set!"
assert os.environ.get("ANTHROPIC_API_KEY"), "❌ ANTHROPIC_API_KEY not set!"
print("✅ API keys verified")
print(f"   OpenAI key:    ...{os.environ['OPENAI_API_KEY'][-6:]}")
print(f"   Anthropic key: ...{os.environ['ANTHROPIC_API_KEY'][-6:]}")

## 2. LLM Service Layer

Wraps OpenAI and Anthropic with token tracking and cost estimation.

In [ ]:
class LLMProvider(Enum):
    OPENAI = "openai"
    ANTHROPIC = "anthropic"


@dataclass
class LLMResponse:
    """Structured response from an LLM call."""
    content: str
    provider: str
    model: str
    input_tokens: int
    output_tokens: int
    total_tokens: int
    latency_seconds: float
    cost_estimate: float


class LLMService:
    """Manages LLM calls with token and cost tracking."""

    PRICING = {
        "gpt-4o-mini": {"input": 0.15, "output": 0.60},
        "gpt-4o": {"input": 2.50, "output": 10.00},
        "claude-3-5-haiku-latest": {"input": 0.80, "output": 4.00},
        "claude-3-5-sonnet-latest": {"input": 3.00, "output": 15.00},
    }

    def __init__(
        self,
        openai_model: str = "gpt-4o-mini",
        anthropic_model: str = "claude-3-5-haiku-latest",
        temperature: float = 0.3,
        max_tokens: int = 512,
    ):
        self.openai_model_name = openai_model
        self.anthropic_model_name = anthropic_model
        self.openai_llm = ChatOpenAI(model=openai_model, temperature=temperature, max_tokens=max_tokens)
        self.anthropic_llm = ChatAnthropic(model=anthropic_model, temperature=temperature, max_tokens=max_tokens)
        self.total_cost = 0.0
        self.call_count = 0
        self.call_log: List[Dict[str, Any]] = []

    def _estimate_cost(self, model: str, inp: int, out: int) -> float:
        p = self.PRICING.get(model, {"input": 1.0, "output": 3.0})
        return (inp * p["input"] + out * p["output"]) / 1_000_000

    def call(
        self,
        system_prompt: str,
        user_message: str,
        provider: LLMProvider = LLMProvider.OPENAI,
    ) -> LLMResponse:
        msgs = [SystemMessage(content=system_prompt), HumanMessage(content=user_message)]
        llm = self.openai_llm if provider == LLMProvider.OPENAI else self.anthropic_llm
        model_name = self.openai_model_name if provider == LLMProvider.OPENAI else self.anthropic_model_name

        start = time.time()
        result = llm.invoke(msgs)
        latency = time.time() - start

        usage = result.usage_metadata or {}
        inp = usage.get("input_tokens", 0)
        out = usage.get("output_tokens", 0)
        cost = self._estimate_cost(model_name, inp, out)
        self.total_cost += cost
        self.call_count += 1
        self.call_log.append({"model": model_name, "input_tokens": inp,
                              "output_tokens": out, "latency": latency, "cost": cost})
        return LLMResponse(
            content=result.content, provider=provider.value, model=model_name,
            input_tokens=inp, output_tokens=out, total_tokens=inp + out,
            latency_seconds=round(latency, 3), cost_estimate=cost,
        )

    def get_cost_summary(self) -> Dict[str, Any]:
        if not self.call_log:
            return {"total_calls": 0, "total_cost": 0.0}
        df = pd.DataFrame(self.call_log)
        return {
            "total_calls": self.call_count,
            "total_cost_usd": round(self.total_cost, 6),
            "avg_latency_s": round(df["latency"].mean(), 3),
            "avg_tokens": int(df["input_tokens"].mean() + df["output_tokens"].mean()),
            "by_model": df.groupby("model")["cost"].sum().to_dict(),
        }


# Use low temperature for judge consistency
llm_service = LLMService(
    openai_model="gpt-4o-mini",
    anthropic_model="claude-3-5-haiku-latest",
    temperature=0.3,
    max_tokens=512,
)

# Smoke test
_r = llm_service.call("You are a helpful assistant.", "Say hello in one sentence.", LLMProvider.OPENAI)
print(f"✅ LLM Service ready — test: {_r.content[:80]}")

## 3. Evaluation Dataset with Human Reference Scores

We define a set of **(query, response)** pairs with **human reference quality scores** on
multiple dimensions. These reference scores serve as ground truth for measuring how well
each LLM judge prompt agrees with human evaluators.

In a real deployment, these would come from expert human annotators. Here we define them
manually to cover a range of quality levels and query types.

In [ ]:
@dataclass
class HumanReferenceScores:
    """Human expert scores for a response on multiple quality dimensions."""
    relevance: float       # 0–1: how relevant to the query
    accuracy: float        # 0–1: factual correctness
    completeness: float    # 0–1: thoroughness of coverage
    clarity: float         # 0–1: readability and coherence
    overall: float         # 0–1: holistic quality

    def mean(self) -> float:
        return float(np.mean([self.relevance, self.accuracy, self.completeness,
                              self.clarity, self.overall]))

    def to_dict(self) -> Dict[str, float]:
        return {"relevance": self.relevance, "accuracy": self.accuracy,
                "completeness": self.completeness, "clarity": self.clarity,
                "overall": self.overall}


@dataclass
class EvalSample:
    """One evaluation sample: a query, a response, and human reference scores."""
    id: str
    query: str
    response: str
    query_type: str  # factual, analytical, creative, coding
    human_scores: HumanReferenceScores


# ── Evaluation dataset ────────────────────────────────────────────────────────
# Each sample has a query, an AI-generated response (of varying quality),
# and human reference scores.

EVAL_DATASET: List[EvalSample] = [
    # ── HIGH quality responses ────────────────────────────────────────────────
    EvalSample(
        id="h1",
        query="What is the CAP theorem in distributed systems?",
        response=(
            "The CAP theorem, formulated by Eric Brewer in 2000 and proved by "
            "Gilbert and Lynch in 2002, states that a distributed data store "
            "cannot simultaneously provide more than two of three guarantees: "
            "Consistency (every read receives the most recent write), "
            "Availability (every request receives a non-error response), and "
            "Partition Tolerance (the system continues to operate despite "
            "network partitions). In practice, since network partitions are "
            "inevitable in distributed systems, the real trade-off is between "
            "consistency and availability. Systems like Google Spanner favour "
            "consistency (CP), while Amazon DynamoDB favours availability (AP)."
        ),
        query_type="factual",
        human_scores=HumanReferenceScores(0.95, 0.95, 0.90, 0.92, 0.93),
    ),
    EvalSample(
        id="h2",
        query="Write a Python function that reverses a linked list.",
        response=(
            "```python\n"
            "class ListNode:\n"
            "    def __init__(self, val=0, next=None):\n"
            "        self.val = val\n"
            "        self.next = next\n\n"
            "def reverse_linked_list(head: ListNode) -> ListNode:\n"
            "    prev = None\n"
            "    current = head\n"
            "    while current:\n"
            "        next_node = current.next\n"
            "        current.next = prev\n"
            "        prev = current\n"
            "        current = next_node\n"
            "    return prev\n"
            "```\n"
            "This iterative approach runs in O(n) time and O(1) space. It works "
            "by maintaining three pointers and reversing each link as we traverse."
        ),
        query_type="coding",
        human_scores=HumanReferenceScores(0.95, 0.98, 0.88, 0.90, 0.93),
    ),
    EvalSample(
        id="h3",
        query="Analyze the trade-offs between REST and GraphQL APIs.",
        response=(
            "REST and GraphQL represent fundamentally different API design philosophies. "
            "REST uses fixed endpoints with predefined data shapes, making it simple, "
            "cacheable, and well-supported by existing HTTP infrastructure. GraphQL "
            "provides a single endpoint with a flexible query language that lets clients "
            "request exactly the data they need. Key trade-offs: (1) Over/under-fetching: "
            "GraphQL eliminates this by design; REST often returns too much or requires "
            "multiple calls. (2) Caching: REST benefits from HTTP caching out of the box; "
            "GraphQL requires custom caching strategies. (3) Complexity: GraphQL shifts "
            "complexity to the server (resolver design, N+1 problems) while REST is "
            "simpler server-side. (4) Tooling: REST has more mature tooling; GraphQL "
            "has better developer experience for frontend teams. For most applications, "
            "REST is the safer default; GraphQL shines when clients have diverse data needs."
        ),
        query_type="analytical",
        human_scores=HumanReferenceScores(0.93, 0.90, 0.92, 0.88, 0.91),
    ),

    # ── MEDIUM quality responses ──────────────────────────────────────────────
    EvalSample(
        id="m1",
        query="Explain how garbage collection works in Java.",
        response=(
            "Java uses garbage collection to automatically manage memory. When objects "
            "are no longer referenced, the garbage collector reclaims their memory. "
            "Java uses a generational approach with young and old generations. "
            "The young generation uses a copying collector, while the old generation "
            "uses a mark-and-sweep approach. There are several GC implementations "
            "like G1, ZGC, and Shenandoah."
        ),
        query_type="factual",
        human_scores=HumanReferenceScores(0.85, 0.80, 0.60, 0.75, 0.72),
    ),
    EvalSample(
        id="m2",
        query="Write a Python decorator that logs function execution time.",
        response=(
            "```python\n"
            "import time\n\n"
            "def timer(func):\n"
            "    def wrapper(*args, **kwargs):\n"
            "        start = time.time()\n"
            "        result = func(*args, **kwargs)\n"
            "        print(f'{func.__name__} took {time.time()-start:.2f}s')\n"
            "        return result\n"
            "    return wrapper\n"
            "```\n"
            "Use `@timer` above any function to log its execution time."
        ),
        query_type="coding",
        human_scores=HumanReferenceScores(0.90, 0.85, 0.55, 0.70, 0.70),
    ),
    EvalSample(
        id="m3",
        query="What are the pros and cons of microservices architecture?",
        response=(
            "Microservices are good because they let teams work independently and "
            "you can scale different parts separately. But they add complexity "
            "with networking, debugging, and deployment. You also need to handle "
            "distributed transactions which is hard. Overall it depends on the "
            "size of your team and application."
        ),
        query_type="analytical",
        human_scores=HumanReferenceScores(0.75, 0.70, 0.50, 0.60, 0.62),
    ),
    EvalSample(
        id="m4",
        query="Describe the concept of eventual consistency.",
        response=(
            "Eventual consistency means that if no new updates are made to a piece "
            "of data, all replicas will eventually converge to the same value. It is "
            "a weaker guarantee than strong consistency but allows for higher "
            "availability and lower latency in distributed systems."
        ),
        query_type="factual",
        human_scores=HumanReferenceScores(0.85, 0.88, 0.55, 0.80, 0.73),
    ),

    # ── LOW quality responses ─────────────────────────────────────────────────
    EvalSample(
        id="l1",
        query="Explain the difference between processes and threads.",
        response=(
            "Processes and threads are both used for running programs. "
            "Processes are heavier and threads are lighter. "
            "You should use threads when you need things to run fast."
        ),
        query_type="factual",
        human_scores=HumanReferenceScores(0.50, 0.40, 0.25, 0.55, 0.38),
    ),
    EvalSample(
        id="l2",
        query="Write a function to check if a string is a palindrome.",
        response=(
            "A palindrome reads the same forwards and backwards, like \"racecar\". "
            "You can check it in Python by comparing the string to its reverse."
        ),
        query_type="coding",
        human_scores=HumanReferenceScores(0.60, 0.70, 0.20, 0.65, 0.40),
    ),
    EvalSample(
        id="l3",
        query="Compare SQL and NoSQL databases.",
        response=(
            "SQL is for structured data and NoSQL is for unstructured data. "
            "SQL is old and NoSQL is new. Use NoSQL for big data."
        ),
        query_type="analytical",
        human_scores=HumanReferenceScores(0.45, 0.35, 0.20, 0.50, 0.30),
    ),
    EvalSample(
        id="l4",
        query="What is a hash table and how does it work?",
        response=(
            "A hash table is a data structure that stores key-value pairs. "
            "It uses hashing."
        ),
        query_type="factual",
        human_scores=HumanReferenceScores(0.55, 0.60, 0.15, 0.50, 0.35),
    ),
]

print(f"✅ Evaluation dataset: {len(EVAL_DATASET)} samples")
print(f"   Quality distribution:")
for sample in EVAL_DATASET:
    label = "HIGH" if sample.human_scores.overall > 0.85 else "MED" if sample.human_scores.overall > 0.55 else "LOW"
    print(f"   [{label:>4}] {sample.id}: overall={sample.human_scores.overall:.2f}  │ {sample.query[:55]}...")

## 4. Judge Prompt Templates (Bandit Arms)

We define **six different LLM judge prompt templates** — each is an *arm* in the bandit. They vary in evaluation methodology, scoring rubric, and reasoning approach. The goal is to find the prompt that produces scores most aligned with human reference labels.

In [ ]:
@dataclass
class JudgePromptTemplate:
    """A judge prompt template (one arm of the bandit)."""
    id: str
    name: str
    system_prompt: str
    description: str


# ── Judge prompt templates ─────────────────────────────────────────────────────

JUDGE_PROMPTS: Dict[str, JudgePromptTemplate] = {
    # ── Arm 1: Structured JSON scorer ─────────────────────────────────────────
    "structured": JudgePromptTemplate(
        id="structured",
        name="Structured JSON Scorer",
        description="Direct scoring on 5 dimensions with JSON output",
        system_prompt=(
            "You are an expert evaluator of AI-generated responses.\n\n"
            "Score the response on each dimension from 0.0 to 1.0:\n"
            "- relevance: How relevant is the response to the query?\n"
            "- accuracy: How factually correct is the response?\n"
            "- completeness: How thorough is the coverage?\n"
            "- clarity: How clear and well-written is it?\n"
            "- overall: Holistic quality assessment.\n\n"
            "Return ONLY valid JSON:\n"
            '{"relevance": <float>, "accuracy": <float>, "completeness": <float>, '
            '"clarity": <float>, "overall": <float>}'
        ),
    ),

    # ── Arm 2: Rubric-based evaluator ─────────────────────────────────────────
    "rubric": JudgePromptTemplate(
        id="rubric",
        name="Rubric-Based Evaluator",
        description="Explicit rubric with grade boundaries for each dimension",
        system_prompt=(
            "You are a meticulous evaluator. Use this rubric:\n\n"
            "RELEVANCE:\n"
            "  0.9-1.0: Directly and fully addresses the query\n"
            "  0.7-0.89: Mostly relevant with minor tangents\n"
            "  0.4-0.69: Partially relevant, misses key aspects\n"
            "  0.0-0.39: Largely irrelevant or off-topic\n\n"
            "ACCURACY:\n"
            "  0.9-1.0: All facts correct, well-sourced\n"
            "  0.7-0.89: Mostly accurate, minor errors\n"
            "  0.4-0.69: Some inaccuracies or misleading claims\n"
            "  0.0-0.39: Major factual errors\n\n"
            "COMPLETENESS:\n"
            "  0.9-1.0: Exhaustive coverage of the topic\n"
            "  0.7-0.89: Covers main points, misses some details\n"
            "  0.4-0.69: Covers basics but lacks depth\n"
            "  0.0-0.39: Superficial or missing key information\n\n"
            "CLARITY:\n"
            "  0.9-1.0: Exceptionally clear and well-structured\n"
            "  0.7-0.89: Clear with good organisation\n"
            "  0.4-0.69: Understandable but could be clearer\n"
            "  0.0-0.39: Confusing or poorly written\n\n"
            "OVERALL: Holistic assessment considering all dimensions.\n\n"
            "Return ONLY valid JSON:\n"
            '{"relevance": <float>, "accuracy": <float>, "completeness": <float>, '
            '"clarity": <float>, "overall": <float>}'
        ),
    ),

    # ── Arm 3: Chain-of-Thought evaluator ─────────────────────────────────────
    "cot": JudgePromptTemplate(
        id="cot",
        name="Chain-of-Thought Evaluator",
        description="Reasons step-by-step before scoring",
        system_prompt=(
            "You are a careful evaluator. Before scoring, think through each dimension step by step.\n\n"
            "Step 1: Assess RELEVANCE - Does the response address the query?\n"
            "Step 2: Assess ACCURACY - Are the facts correct?\n"
            "Step 3: Assess COMPLETENESS - Is the coverage thorough?\n"
            "Step 4: Assess CLARITY - Is it well-written and clear?\n"
            "Step 5: Determine OVERALL quality.\n\n"
            "After your reasoning, return scores as ONLY valid JSON on the last line:\n"
            '{"relevance": <float>, "accuracy": <float>, "completeness": <float>, '
            '"clarity": <float>, "overall": <float>}\n\n'
            "Each score must be between 0.0 and 1.0."
        ),
    ),

    # ── Arm 4: Reference-anchored evaluator ───────────────────────────────────
    "anchored": JudgePromptTemplate(
        id="anchored",
        name="Reference-Anchored Evaluator",
        description="Provides concrete reference examples for calibration",
        system_prompt=(
            "You are an evaluator. Use these reference anchors for calibration:\n\n"
            "EXAMPLE OF 0.9+ QUALITY: A response that is comprehensive, factually correct, "
            "well-structured with examples, and directly addresses every aspect of the query.\n\n"
            "EXAMPLE OF 0.6-0.8 QUALITY: A response that addresses the query but misses some "
            "details, may have minor inaccuracies, or lacks depth.\n\n"
            "EXAMPLE OF 0.3-0.5 QUALITY: A response that is superficial, has factual errors, "
            "or only partially addresses the query.\n\n"
            "EXAMPLE OF <0.3 QUALITY: A response that is largely irrelevant, contains major "
            "errors, or fails to address the query.\n\n"
            "Score the response on: relevance, accuracy, completeness, clarity, overall.\n"
            "Each score 0.0 to 1.0.\n\n"
            "Return ONLY valid JSON:\n"
            '{"relevance": <float>, "accuracy": <float>, "completeness": <float>, '
            '"clarity": <float>, "overall": <float>}'
        ),
    ),

    # ── Arm 5: Checklist evaluator ─────────────────────────────────────────────
    "checklist": JudgePromptTemplate(
        id="checklist",
        name="Checklist Evaluator",
        description="Evaluates against a checklist of quality criteria",
        system_prompt=(
            "You are an evaluator. For each dimension, check the criteria below and assign a score.\n\n"
            "RELEVANCE checklist:\n"
            "  [ ] Addresses the main question\n"
            "  [ ] Stays on topic throughout\n"
            "  [ ] Answers what was specifically asked\n\n"
            "ACCURACY checklist:\n"
            "  [ ] Key facts are correct\n"
            "  [ ] No misleading statements\n"
            "  [ ] Technical terms used correctly\n\n"
            "COMPLETENESS checklist:\n"
            "  [ ] Covers the main concept\n"
            "  [ ] Includes relevant details\n"
            "  [ ] Provides examples or context\n\n"
            "CLARITY checklist:\n"
            "  [ ] Well-organised structure\n"
            "  [ ] Clear language\n"
            "  [ ] Logical flow\n\n"
            "Score each dimension 0.0-1.0 based on how many criteria are met.\n"
            "OVERALL: holistic quality.\n\n"
            "Return ONLY valid JSON:\n"
            '{"relevance": <float>, "accuracy": <float>, "completeness": <float>, '
            '"clarity": <float>, "overall": <float>}'
        ),
    ),

    # ── Arm 6: Strict critic ───────────────────────────────────────────────────
    "critic": JudgePromptTemplate(
        id="critic",
        name="Strict Critic",
        description="Harsh evaluator that penalises aggressively for any weakness",
        system_prompt=(
            "You are a strict, demanding critic. You have very high standards.\n\n"
            "A score of 1.0 should be almost never given. Penalise harshly for:\n"
            "- Any factual inaccuracy (accuracy score drops significantly)\n"
            "- Missing important information (completeness drops)\n"
            "- Vague or hand-wavy statements (clarity drops)\n"
            "- Not directly answering the question (relevance drops)\n\n"
            "Be strict but fair. Score each dimension 0.0-1.0:\n"
            "relevance, accuracy, completeness, clarity, overall.\n\n"
            "Return ONLY valid JSON:\n"
            '{"relevance": <float>, "accuracy": <float>, "completeness": <float>, '
            '"clarity": <float>, "overall": <float>}'
        ),
    ),
}

print(f"✅ Defined {len(JUDGE_PROMPTS)} judge prompt templates (bandit arms):")
for jid, j in JUDGE_PROMPTS.items():
    print(f"   🎯 {j.name} ({jid}): {j.description}")

## 5. Judge Execution and Agreement Metrics

Execute a judge prompt against an evaluation sample and measure agreement with human scores.

In [ ]:
class JudgeScores(BaseModel):
    """Pydantic model for parsed judge output."""
    relevance: float = Field(ge=0, le=1)
    accuracy: float = Field(ge=0, le=1)
    completeness: float = Field(ge=0, le=1)
    clarity: float = Field(ge=0, le=1)
    overall: float = Field(ge=0, le=1)


def run_judge(
    llm_service: LLMService,
    judge_template: JudgePromptTemplate,
    sample: EvalSample,
    provider: LLMProvider = LLMProvider.OPENAI,
) -> Tuple[JudgeScores, LLMResponse]:
    """
    Run one judge prompt against one evaluation sample.
    Returns parsed scores and the raw LLM response.
    """
    user_message = (
        f"## Query\n{sample.query}\n\n"
        f"## Response to Evaluate\n{sample.response}\n\n"
        f"Evaluate now."
    )

    resp = llm_service.call(judge_template.system_prompt, user_message, provider)

    try:
        raw = resp.content.strip()
        # Handle markdown code fences
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        # Handle CoT prompt: extract last JSON block
        if "{" in raw:
            last_json_start = raw.rfind("{")
            last_json_end = raw.rfind("}") + 1
            raw = raw[last_json_start:last_json_end]
        scores = JudgeScores(**json.loads(raw))
    except Exception:
        scores = JudgeScores(relevance=0.5, accuracy=0.5, completeness=0.5,
                             clarity=0.5, overall=0.5)

    return scores, resp


def compute_agreement(judge_scores: JudgeScores, human_scores: HumanReferenceScores) -> float:
    """
    Compute agreement between judge and human scores.

    Uses 1 - Mean Absolute Error (MAE) across all dimensions.
    A score of 1.0 means perfect agreement; 0.0 means maximum disagreement.
    """
    dims = ["relevance", "accuracy", "completeness", "clarity", "overall"]
    judge_vals = [getattr(judge_scores, d) for d in dims]
    human_vals = [getattr(human_scores, d) for d in dims]
    mae = np.mean(np.abs(np.array(judge_vals) - np.array(human_vals)))
    return float(1.0 - mae)


def compute_rank_agreement(judge_scores: JudgeScores, human_scores: HumanReferenceScores) -> float:
    """
    Measure how well the judge preserves the ranking of dimensions.
    Uses Spearman rank correlation (normalised to 0-1).
    """
    from scipy.stats import spearmanr
    dims = ["relevance", "accuracy", "completeness", "clarity", "overall"]
    judge_vals = [getattr(judge_scores, d) for d in dims]
    human_vals = [getattr(human_scores, d) for d in dims]
    corr, _ = spearmanr(judge_vals, human_vals)
    if np.isnan(corr):
        return 0.5
    return float((corr + 1) / 2)  # normalise from [-1, 1] to [0, 1]


def compute_reward(
    judge_scores: JudgeScores,
    human_scores: HumanReferenceScores,
    cost: float,
    latency: float,
    cost_weight: float = 0.05,
    latency_weight: float = 0.02,
) -> float:
    """
    Compute bandit reward: primarily agreement with human scores,
    with small penalties for cost and latency.
    """
    agreement = compute_agreement(judge_scores, human_scores)
    rank_agreement = compute_rank_agreement(judge_scores, human_scores)
    quality = 0.7 * agreement + 0.3 * rank_agreement
    norm_cost = min(cost / 0.002, 1.0)
    norm_lat = min(latency / 5.0, 1.0)
    reward = quality - cost_weight * norm_cost - latency_weight * norm_lat
    return float(np.clip(reward, 0, 1))


# ── Quick test ─────────────────────────────────────────────────────────────────
test_scores, test_resp = run_judge(llm_service, JUDGE_PROMPTS["structured"], EVAL_DATASET[0])
test_reward = compute_reward(test_scores, EVAL_DATASET[0].human_scores,
                             test_resp.cost_estimate, test_resp.latency_seconds)
print(f"✅ Judge execution test")
print(f"   Judge scores:   {test_scores.model_dump()}")
print(f"   Human scores:   {EVAL_DATASET[0].human_scores.to_dict()}")
print(f"   Agreement:      {compute_agreement(test_scores, EVAL_DATASET[0].human_scores):.3f}")
print(f"   Reward:         {test_reward:.3f}")

## 6. Multi-Armed Bandit Algorithms

Three classic bandit algorithms to learn which judge prompt best agrees with human scores.

In [ ]:
class MultiArmedBandit(ABC):
    """Abstract base class for bandit algorithms."""

    def __init__(self, n_arms: int, arm_names: List[str]):
        self.n_arms = n_arms
        self.arm_names = arm_names
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)
        self.history: List[Dict[str, Any]] = []

    @abstractmethod
    def select_arm(self) -> int:
        pass

    def update(self, arm: int, reward: float):
        self.counts[arm] += 1
        n = self.counts[arm]
        self.values[arm] += (reward - self.values[arm]) / n
        self.history.append({"arm": arm, "arm_name": self.arm_names[arm], "reward": reward})

    def get_best_arm(self) -> int:
        return int(np.argmax(self.values))

    def get_summary(self) -> pd.DataFrame:
        return pd.DataFrame({
            "arm": self.arm_names,
            "pulls": self.counts.astype(int),
            "mean_reward": np.round(self.values, 4),
        })


class EpsilonGreedyBandit(MultiArmedBandit):
    """Epsilon-Greedy: explore randomly with probability epsilon."""

    def __init__(self, n_arms: int, arm_names: List[str], epsilon: float = 0.15):
        super().__init__(n_arms, arm_names)
        self.epsilon = epsilon

    def select_arm(self) -> int:
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_arms)
        return self.get_best_arm()


class UCBBandit(MultiArmedBandit):
    """Upper Confidence Bound: balance exploration and exploitation analytically."""

    def __init__(self, n_arms: int, arm_names: List[str], c: float = 2.0):
        super().__init__(n_arms, arm_names)
        self.c = c
        self.total_counts = 0

    def select_arm(self) -> int:
        self.total_counts += 1
        for arm in range(self.n_arms):
            if self.counts[arm] == 0:
                return arm
        ucb = self.values + self.c * np.sqrt(np.log(self.total_counts) / self.counts)
        return int(np.argmax(ucb))


class ThompsonSamplingBandit(MultiArmedBandit):
    """Thompson Sampling: Bayesian approach with Beta posteriors."""

    def __init__(self, n_arms: int, arm_names: List[str]):
        super().__init__(n_arms, arm_names)
        self.alpha = np.ones(n_arms)  # Beta posterior successes
        self.beta_param = np.ones(n_arms)  # Beta posterior failures

    def select_arm(self) -> int:
        samples = np.random.beta(self.alpha, self.beta_param)
        return int(np.argmax(samples))

    def update(self, arm: int, reward: float):
        super().update(arm, reward)
        # Use reward as probability of success
        if reward > 0.5:
            self.alpha[arm] += reward
        else:
            self.beta_param[arm] += (1 - reward)


arm_names = list(JUDGE_PROMPTS.keys())
n_arms = len(arm_names)
print(f"✅ Bandit algorithms defined: Epsilon-Greedy, UCB, Thompson Sampling")
print(f"   Number of arms: {n_arms} ({', '.join(arm_names)})")

## 7. LangGraph Workflow: Bandit Evaluation Loop

We use LangGraph to orchestrate the bandit evaluation loop as a state machine:

```
START → pick_sample → select_judge → run_judge → compute_reward → (loop or END)
```

Each iteration:
1. Picks a random evaluation sample from the dataset
2. The bandit selects a judge prompt template
3. Runs the selected judge against the sample (real LLM call)
4. Computes agreement with human scores as reward
5. Updates the bandit's beliefs

In [ ]:
class BanditWorkflowState(TypedDict):
    """LangGraph state for the bandit evaluation loop."""
    iteration: int
    max_iterations: int
    # Current sample
    sample_id: str
    sample_query: str
    sample_response: str
    sample_query_type: str
    human_overall: float
    # Judge selection
    selected_judge_id: str
    arm_idx: int
    # Judge output
    judge_relevance: float
    judge_accuracy: float
    judge_completeness: float
    judge_clarity: float
    judge_overall: float
    # Reward
    agreement: float
    reward: float
    cost: float
    latency: float
    # Accumulated logs
    rewards_log: List[float]
    details_log: List[Dict[str, Any]]


def build_bandit_workflow(
    bandit: MultiArmedBandit,
    llm_svc: LLMService,
    judge_prompts: Dict[str, JudgePromptTemplate],
    eval_dataset: List[EvalSample],
    provider: LLMProvider = LLMProvider.OPENAI,
):
    """
    Build a LangGraph state machine for the bandit evaluation loop.
    """
    arm_ids = list(judge_prompts.keys())

    def pick_sample(state: BanditWorkflowState) -> dict:
        """Randomly sample an evaluation example."""
        sample = random.choice(eval_dataset)
        return {
            "sample_id": sample.id,
            "sample_query": sample.query,
            "sample_response": sample.response,
            "sample_query_type": sample.query_type,
            "human_overall": sample.human_scores.overall,
        }

    def select_judge(state: BanditWorkflowState) -> dict:
        """Bandit selects which judge prompt to use."""
        arm_idx = bandit.select_arm()
        return {
            "selected_judge_id": arm_ids[arm_idx],
            "arm_idx": arm_idx,
        }

    def run_judge_node(state: BanditWorkflowState) -> dict:
        """Run the selected judge prompt against the sample (real LLM call)."""
        judge_template = judge_prompts[state["selected_judge_id"]]
        sample = next(s for s in eval_dataset if s.id == state["sample_id"])
        scores, resp = run_judge(llm_svc, judge_template, sample, provider)
        return {
            "judge_relevance": scores.relevance,
            "judge_accuracy": scores.accuracy,
            "judge_completeness": scores.completeness,
            "judge_clarity": scores.clarity,
            "judge_overall": scores.overall,
            "cost": resp.cost_estimate,
            "latency": resp.latency_seconds,
        }

    def compute_reward_node(state: BanditWorkflowState) -> dict:
        """Compute agreement reward and update bandit."""
        sample = next(s for s in eval_dataset if s.id == state["sample_id"])
        judge_scores = JudgeScores(
            relevance=state["judge_relevance"],
            accuracy=state["judge_accuracy"],
            completeness=state["judge_completeness"],
            clarity=state["judge_clarity"],
            overall=state["judge_overall"],
        )
        agreement = compute_agreement(judge_scores, sample.human_scores)
        reward = compute_reward(judge_scores, sample.human_scores,
                                state["cost"], state["latency"])

        # Update bandit
        bandit.update(state["arm_idx"], reward)

        prev_rewards = state.get("rewards_log", []) or []
        prev_details = state.get("details_log", []) or []

        it = state["iteration"] + 1
        print(f"   [{it}/{state['max_iterations']}] judge={state['selected_judge_id']:<12s} "
              f"agreement={agreement:.3f} reward={reward:.3f} "
              f"sample={state['sample_id']}")

        return {
            "agreement": agreement,
            "reward": reward,
            "iteration": it,
            "rewards_log": prev_rewards + [reward],
            "details_log": prev_details + [{
                "iteration": state["iteration"],
                "sample_id": state["sample_id"],
                "sample_query_type": state["sample_query_type"],
                "human_overall": state["human_overall"],
                "judge_id": state["selected_judge_id"],
                "judge_overall": state["judge_overall"],
                "agreement": agreement,
                "reward": reward,
                "cost": state["cost"],
                "latency": state["latency"],
            }],
        }

    def should_continue(state: BanditWorkflowState) -> str:
        if state["iteration"] >= state["max_iterations"]:
            return END
        return "pick_sample"

    # ── Build graph ────────────────────────────────────────────────────────────
    workflow = StateGraph(BanditWorkflowState)

    workflow.add_node("pick_sample", pick_sample)
    workflow.add_node("select_judge", select_judge)
    workflow.add_node("run_judge", run_judge_node)
    workflow.add_node("compute_reward", compute_reward_node)

    workflow.add_edge(START, "pick_sample")
    workflow.add_edge("pick_sample", "select_judge")
    workflow.add_edge("select_judge", "run_judge")
    workflow.add_edge("run_judge", "compute_reward")
    workflow.add_conditional_edges("compute_reward", should_continue,
                                   {END: END, "pick_sample": "pick_sample"})

    return workflow.compile()


print("✅ LangGraph bandit workflow builder ready")

## 8. Training: Finding the Best Judge Prompt

We train all three bandit algorithms side by side. Each iteration makes **1 real LLM call** (the judge evaluation). With 40 iterations per algorithm and `gpt-4o-mini`, estimated cost is < $0.05.

In [ ]:
N_ITERATIONS = 40

INITIAL_STATE: BanditWorkflowState = {
    "iteration": 0,
    "max_iterations": N_ITERATIONS,
    "sample_id": "", "sample_query": "", "sample_response": "",
    "sample_query_type": "", "human_overall": 0.0,
    "selected_judge_id": "", "arm_idx": 0,
    "judge_relevance": 0.0, "judge_accuracy": 0.0,
    "judge_completeness": 0.0, "judge_clarity": 0.0,
    "judge_overall": 0.0,
    "agreement": 0.0, "reward": 0.0,
    "cost": 0.0, "latency": 0.0,
    "rewards_log": [],
    "details_log": [],
}

print(f"📋 Training plan: {N_ITERATIONS} iterations x 3 algorithms = {N_ITERATIONS * 3} LLM calls")
print(f"   Estimated cost: ~${N_ITERATIONS * 3 * 0.0004:.2f} (gpt-4o-mini)")

In [ ]:
# ── Train Thompson Sampling ────────────────────────────────────────────────────
print("\n\n🎰 Thompson Sampling")
print("=" * 60)

ts_bandit = ThompsonSamplingBandit(n_arms, arm_names)
ts_workflow = build_bandit_workflow(ts_bandit, llm_service, JUDGE_PROMPTS, EVAL_DATASET)

ts_result = ts_workflow.invoke(dict(INITIAL_STATE))
ts_rewards = ts_result["rewards_log"]
ts_details = ts_result["details_log"]

print(f"\n✅ Thompson Sampling complete: mean reward = {np.mean(ts_rewards):.4f}")
print(ts_bandit.get_summary().to_string(index=False))

In [ ]:
# ── Train UCB ──────────────────────────────────────────────────────────────────
print("\n\n📊 UCB (Upper Confidence Bound)")
print("=" * 60)

ucb_bandit = UCBBandit(n_arms, arm_names, c=2.0)
ucb_workflow = build_bandit_workflow(ucb_bandit, llm_service, JUDGE_PROMPTS, EVAL_DATASET)

ucb_result = ucb_workflow.invoke(dict(INITIAL_STATE))
ucb_rewards = ucb_result["rewards_log"]
ucb_details = ucb_result["details_log"]

print(f"\n✅ UCB complete: mean reward = {np.mean(ucb_rewards):.4f}")
print(ucb_bandit.get_summary().to_string(index=False))

In [ ]:
# ── Train Epsilon-Greedy ───────────────────────────────────────────────────────
print("\n\n🎲 Epsilon-Greedy")
print("=" * 60)

eg_bandit = EpsilonGreedyBandit(n_arms, arm_names, epsilon=0.15)
eg_workflow = build_bandit_workflow(eg_bandit, llm_service, JUDGE_PROMPTS, EVAL_DATASET)

eg_result = eg_workflow.invoke(dict(INITIAL_STATE))
eg_rewards = eg_result["rewards_log"]
eg_details = eg_result["details_log"]

print(f"\n✅ Epsilon-Greedy complete: mean reward = {np.mean(eg_rewards):.4f}")
print(eg_bandit.get_summary().to_string(index=False))

## 9. Training Visualisation

In [ ]:
def moving_avg(data, window=5):
    return pd.Series(data).rolling(window=window, min_periods=1).mean()


fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("LLM-as-Judge Prompt Selection — Multi-Armed Bandit Training",
             fontsize=16, fontweight="bold")

# ── 1. Reward curves ─────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(moving_avg(ts_rewards), label="Thompson Sampling", linewidth=2)
ax.plot(moving_avg(ucb_rewards), label="UCB", linewidth=2)
ax.plot(moving_avg(eg_rewards), label="Epsilon-Greedy", linewidth=2)
ax.set_xlabel("Iteration")
ax.set_ylabel("Reward (Moving Avg)")
ax.set_title("Reward Curves")
ax.legend()
ax.grid(True, alpha=0.3)

# ── 2. Mean reward comparison ────────────────────────────────────────────────
ax = axes[0, 1]
algo_names = ["Thompson\nSampling", "UCB", "Epsilon\nGreedy"]
algo_means = [np.mean(ts_rewards), np.mean(ucb_rewards), np.mean(eg_rewards)]
algo_stds = [np.std(ts_rewards), np.std(ucb_rewards), np.std(eg_rewards)]
colors = ["#2ecc71", "#3498db", "#e74c3c"]
bars = ax.bar(algo_names, algo_means, yerr=algo_stds, color=colors,
              edgecolor="black", capsize=5)
for bar, val in zip(bars, algo_means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.3f}", ha="center", fontsize=11)
ax.set_ylabel("Mean Reward")
ax.set_title("Algorithm Comparison")
ax.grid(True, alpha=0.3, axis="y")

# ── 3. Arm pull distribution ─────────────────────────────────────────────────
ax = axes[1, 0]
summaries = [
    ("Thompson\nSampling", ts_bandit),
    ("UCB", ucb_bandit),
    ("Epsilon\nGreedy", eg_bandit),
]
x = np.arange(n_arms)
width = 0.25
for i, (name, b) in enumerate(summaries):
    ax.bar(x + i * width, b.counts, width, label=name, alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels(arm_names, rotation=30, ha="right")
ax.set_ylabel("Number of Pulls")
ax.set_title("Arm Pull Distribution")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

# ── 4. Mean reward per judge prompt (Thompson Sampling) ──────────────────────
ax = axes[1, 1]
ts_summary = ts_bandit.get_summary()
prompt_colors = plt.cm.Set2(np.linspace(0, 1, n_arms))
bars = ax.barh(ts_summary["arm"], ts_summary["mean_reward"], color=prompt_colors,
               edgecolor="black")
for bar, val in zip(bars, ts_summary["mean_reward"]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=10)
ax.set_xlabel("Mean Reward")
ax.set_title("Thompson Sampling: Reward per Judge Prompt")
ax.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

## 10. Detailed Agreement Analysis

Analyse how each judge prompt performs across different quality levels and query types.

In [ ]:
# Combine all detail logs
all_details = ts_details + ucb_details + eg_details
df_all = pd.DataFrame(all_details)

# ── Agreement by judge prompt ──────────────────────────────────────────────────
print("\n📊 Agreement by Judge Prompt (across all algorithms)")
print("=" * 60)
agg = df_all.groupby("judge_id").agg(
    n_pulls=("reward", "count"),
    mean_agreement=("agreement", "mean"),
    mean_reward=("reward", "mean"),
    std_reward=("reward", "std"),
    mean_cost=("cost", "mean"),
    mean_latency=("latency", "mean"),
).round(4).sort_values("mean_reward", ascending=False)
print(agg.to_string())

# ── Agreement by query type ────────────────────────────────────────────────────
print("\n\n📊 Agreement by Query Type")
print("=" * 60)
qt_agg = df_all.groupby("sample_query_type").agg(
    n_samples=("reward", "count"),
    mean_agreement=("agreement", "mean"),
    mean_reward=("reward", "mean"),
).round(4)
print(qt_agg.to_string())

In [ ]:
# ── Heatmap: Judge prompt x Quality tier ──────────────────────────────────────
df_all["quality_tier"] = df_all["human_overall"].apply(
    lambda x: "HIGH (>0.85)" if x > 0.85 else "MED (0.55-0.85)" if x > 0.55 else "LOW (<0.55)"
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap: Judge x Quality tier
ax = axes[0]
pivot = df_all.pivot_table(values="agreement", index="judge_id",
                           columns="quality_tier", aggfunc="mean")
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax, vmin=0.5, vmax=1.0)
ax.set_title("Agreement by Judge Prompt \u00d7 Quality Tier")
ax.set_ylabel("Judge Prompt")
ax.set_xlabel("Response Quality Tier")

# Heatmap: Judge x Query type
ax = axes[1]
pivot2 = df_all.pivot_table(values="agreement", index="judge_id",
                            columns="sample_query_type", aggfunc="mean")
sns.heatmap(pivot2, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax, vmin=0.5, vmax=1.0)
ax.set_title("Agreement by Judge Prompt \u00d7 Query Type")
ax.set_ylabel("Judge Prompt")
ax.set_xlabel("Query Type")

plt.tight_layout()
plt.show()

## 11. Head-to-Head: Best Judge vs Worst on Full Dataset

Run the best-performing judge prompt (selected by Thompson Sampling) alongside
the worst-performing prompt on the full dataset to compare agreement.

In [ ]:
best_arm_idx = ts_bandit.get_best_arm()
worst_arm_idx = int(np.argmin(ts_bandit.values))
best_judge_id = arm_names[best_arm_idx]
worst_judge_id = arm_names[worst_arm_idx]

print(f"🏆 Best judge prompt:  {best_judge_id} ({JUDGE_PROMPTS[best_judge_id].name})")
print(f"   Mean reward: {ts_bandit.values[best_arm_idx]:.4f}")
print(f"\n🥄 Worst judge prompt: {worst_judge_id} ({JUDGE_PROMPTS[worst_judge_id].name})")
print(f"   Mean reward: {ts_bandit.values[worst_arm_idx]:.4f}")

# Run both on all samples
print(f"\n🔍 Running head-to-head on all {len(EVAL_DATASET)} samples...")
print("=" * 70)

h2h_results = []
for sample in EVAL_DATASET:
    # Best judge
    best_scores, best_resp = run_judge(llm_service, JUDGE_PROMPTS[best_judge_id], sample)
    best_agreement = compute_agreement(best_scores, sample.human_scores)

    # Worst judge
    worst_scores, worst_resp = run_judge(llm_service, JUDGE_PROMPTS[worst_judge_id], sample)
    worst_agreement = compute_agreement(worst_scores, sample.human_scores)

    h2h_results.append({
        "sample": sample.id,
        "query_type": sample.query_type,
        "human_overall": sample.human_scores.overall,
        "best_agreement": best_agreement,
        "best_overall": best_scores.overall,
        "worst_agreement": worst_agreement,
        "worst_overall": worst_scores.overall,
    })

    tier = "HIGH" if sample.human_scores.overall > 0.85 else "MED" if sample.human_scores.overall > 0.55 else "LOW"
    print(f"   [{tier:>4}] {sample.id}: "
          f"best({best_judge_id})={best_agreement:.3f} "
          f"worst({worst_judge_id})={worst_agreement:.3f} "
          f"\u0394={best_agreement - worst_agreement:+.3f}")

h2h_df = pd.DataFrame(h2h_results)

print(f"\n———")
print(f"Best mean agreement:  {h2h_df['best_agreement'].mean():.4f}")
print(f"Worst mean agreement: {h2h_df['worst_agreement'].mean():.4f}")
print(f"Improvement:          {h2h_df['best_agreement'].mean() - h2h_df['worst_agreement'].mean():+.4f}")

## 12. Cross-Provider Comparison

Run the best judge prompt with both OpenAI and Anthropic to check if the judge prompt's
quality is stable across providers.

In [ ]:
print(f"\n🔀 Cross-Provider: Running '{best_judge_id}' on both OpenAI and Anthropic")
print("=" * 70)

cross_provider_results = []
# Use a subset to keep costs down
subset = EVAL_DATASET[:5]

for sample in subset:
    # OpenAI
    oai_scores, oai_resp = run_judge(llm_service, JUDGE_PROMPTS[best_judge_id],
                                     sample, LLMProvider.OPENAI)
    oai_agreement = compute_agreement(oai_scores, sample.human_scores)

    # Anthropic
    ant_scores, ant_resp = run_judge(llm_service, JUDGE_PROMPTS[best_judge_id],
                                     sample, LLMProvider.ANTHROPIC)
    ant_agreement = compute_agreement(ant_scores, sample.human_scores)

    cross_provider_results.append({
        "sample": sample.id,
        "openai_agreement": oai_agreement,
        "anthropic_agreement": ant_agreement,
        "openai_overall": oai_scores.overall,
        "anthropic_overall": ant_scores.overall,
        "human_overall": sample.human_scores.overall,
        "openai_cost": oai_resp.cost_estimate,
        "anthropic_cost": ant_resp.cost_estimate,
    })

    print(f"   {sample.id}: OpenAI={oai_agreement:.3f} vs Anthropic={ant_agreement:.3f} "
          f"(human={sample.human_scores.overall:.2f})")

cp_df = pd.DataFrame(cross_provider_results)
print(f"\nMean agreement — OpenAI:    {cp_df['openai_agreement'].mean():.4f}")
print(f"Mean agreement — Anthropic: {cp_df['anthropic_agreement'].mean():.4f}")
print(f"Mean cost — OpenAI:    ${cp_df['openai_cost'].mean():.6f}")
print(f"Mean cost — Anthropic: ${cp_df['anthropic_cost'].mean():.6f}")

## 13. Production Integration: Adaptive Judge Selector

A production-ready class that wraps a trained bandit to select the best judge prompt
for each evaluation. Supports online learning from human feedback.

In [ ]:
class AdaptiveJudgeSelector:
    """
    Production wrapper for adaptive LLM-as-Judge prompt selection.

    Uses a trained bandit to select the best judge prompt, with optional
    online learning from human feedback.
    """

    def __init__(
        self,
        llm_service: LLMService,
        judge_prompts: Dict[str, JudgePromptTemplate],
        pretrained_bandit: ThompsonSamplingBandit = None,
        provider: LLMProvider = LLMProvider.OPENAI,
        online_learning: bool = True,
    ):
        self.llm_service = llm_service
        self.judge_prompts = judge_prompts
        self.arm_ids = list(judge_prompts.keys())
        self.provider = provider
        self.online_learning = online_learning
        self.evaluation_log: List[Dict[str, Any]] = []

        if pretrained_bandit:
            self.bandit = pretrained_bandit
        else:
            self.bandit = ThompsonSamplingBandit(len(self.arm_ids), self.arm_ids)

    def evaluate(
        self,
        query: str,
        response: str,
        query_type: str = "general",
        human_scores: Optional[HumanReferenceScores] = None,
    ) -> Dict[str, Any]:
        """
        Evaluate a response using the bandit-selected judge prompt.

        If human_scores are provided (online learning), the bandit is updated.
        """
        # 1. Select judge
        arm_idx = self.bandit.select_arm()
        judge_id = self.arm_ids[arm_idx]
        judge_template = self.judge_prompts[judge_id]

        # 2. Run judge
        sample = EvalSample(
            id="live", query=query, response=response,
            query_type=query_type,
            human_scores=human_scores or HumanReferenceScores(0.5, 0.5, 0.5, 0.5, 0.5),
        )
        scores, resp = run_judge(self.llm_service, judge_template, sample, self.provider)

        result = {
            "judge_prompt": judge_id,
            "judge_name": judge_template.name,
            "scores": {
                "relevance": scores.relevance,
                "accuracy": scores.accuracy,
                "completeness": scores.completeness,
                "clarity": scores.clarity,
                "overall": scores.overall,
            },
            "tokens": resp.total_tokens,
            "latency": resp.latency_seconds,
            "cost": resp.cost_estimate,
        }

        # 3. Online learning
        if human_scores and self.online_learning:
            agreement = compute_agreement(scores, human_scores)
            reward = compute_reward(scores, human_scores, resp.cost_estimate,
                                    resp.latency_seconds)
            self.bandit.update(arm_idx, reward)
            result["agreement"] = agreement
            result["reward"] = reward

        self.evaluation_log.append(result)
        return result

    def record_human_feedback(
        self,
        judge_id: str,
        judge_scores: JudgeScores,
        human_scores: HumanReferenceScores,
    ):
        """Record delayed human feedback for a past evaluation."""
        arm_idx = self.arm_ids.index(judge_id)
        agreement = compute_agreement(judge_scores, human_scores)
        self.bandit.update(arm_idx, agreement)

    def get_best_judge(self) -> str:
        """Return the currently best-performing judge prompt."""
        return self.arm_ids[self.bandit.get_best_arm()]

    def get_stats(self) -> Dict[str, Any]:
        if not self.evaluation_log:
            return {"message": "No evaluations yet"}
        df = pd.DataFrame(self.evaluation_log)
        stats = {
            "total_evaluations": len(df),
            "total_cost_usd": round(df["cost"].sum(), 6),
            "avg_latency_s": round(df["latency"].mean(), 3),
            "best_judge": self.get_best_judge(),
            "bandit_summary": self.bandit.get_summary().to_dict(orient="records"),
        }
        if "reward" in df.columns:
            stats["avg_reward"] = round(df["reward"].dropna().mean(), 4)
        return stats


# ── Instantiate with trained bandit ────────────────────────────────────────────
selector = AdaptiveJudgeSelector(
    llm_service=llm_service,
    judge_prompts=JUDGE_PROMPTS,
    pretrained_bandit=ts_bandit,
    provider=LLMProvider.OPENAI,
    online_learning=True,
)

# ── Demo evaluation ───────────────────────────────────────────────────────────
demo_result = selector.evaluate(
    query="Explain the difference between concurrency and parallelism.",
    response=(
        "Concurrency is about dealing with multiple things at once (structure), "
        "while parallelism is about doing multiple things at once (execution). "
        "Concurrency can exist on a single core through interleaving, while "
        "parallelism requires multiple cores. Go uses goroutines for concurrency; "
        "Python's multiprocessing module enables parallelism."
    ),
    query_type="factual",
)

print("🚀 Production Adaptive Judge Selector — Demo")
print("=" * 60)
print(f"Selected judge: {demo_result['judge_prompt']} ({demo_result['judge_name']})")
print(f"Scores: {demo_result['scores']}")
print(f"Tokens: {demo_result['tokens']} | Latency: {demo_result['latency']}s | "
      f"Cost: ${demo_result['cost']:.6f}")
print(f"\nBest judge overall: {selector.get_best_judge()}")

## 14. Cost and Performance Summary

In [ ]:
# ── Cost breakdown ────────────────────────────────────────────────────────────
cost_summary = llm_service.get_cost_summary()
print("💰 LLM Cost Summary")
print("=" * 40)
for k, v in cost_summary.items():
    print(f"  {k:20s} {v}")

# ── Final bandit summaries ────────────────────────────────────────────────────
print("\n\n🏆 Final Bandit Summaries")
print("=" * 60)
for name, b in [("Thompson Sampling", ts_bandit), ("UCB", ucb_bandit),
                 ("Epsilon-Greedy", eg_bandit)]:
    best = arm_names[b.get_best_arm()]
    print(f"\n{name}:")
    print(f"  Best judge prompt: {best} ({JUDGE_PROMPTS[best].name})")
    print(f"  Mean reward:       {b.values[b.get_best_arm()]:.4f}")
    print(b.get_summary().to_string(index=False))

In [ ]:
# ── Final visualisation: Thompson Sampling posteriors ──────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

x = np.linspace(0, 1, 200)
from scipy.stats import beta as beta_dist

for i, name in enumerate(arm_names):
    a = ts_bandit.alpha[i]
    b = ts_bandit.beta_param[i]
    y = beta_dist.pdf(x, a, b)
    ax.plot(x, y, linewidth=2, label=f"{name} (\u03b1={a:.1f}, \u03b2={b:.1f})")
    ax.fill_between(x, y, alpha=0.1)

ax.set_xlabel("Reward Probability")
ax.set_ylabel("Density")
ax.set_title("Thompson Sampling: Beta Posterior Distributions per Judge Prompt")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 15. Summary

This notebook demonstrated **LLM-as-Judge prompt selection** using **multi-armed bandits**:

| Component | Implementation |
|---|---|
| **Arms** | 6 judge prompt templates (structured, rubric, CoT, anchored, checklist, critic) |
| **Reward Signal** | Agreement with human reference scores (MAE + Spearman rank correlation) |
| **Bandit Algorithms** | Thompson Sampling, UCB, Epsilon-Greedy |
| **Workflow Engine** | LangGraph state machine (pick_sample → select_judge → run_judge → compute_reward → loop) |
| **LLM Providers** | OpenAI (gpt-4o-mini), Anthropic (claude-3.5-haiku) via LangChain |
| **Evaluation Dimensions** | Relevance, accuracy, completeness, clarity, overall |
| **Cost Tracking** | Real token usage and USD cost estimation |
| **Production** | `AdaptiveJudgeSelector` with online learning and human feedback |

### Key Takeaways

1. **Different judge prompts produce meaningfully different agreement levels** with human scores — prompt design matters.
2. **Thompson Sampling converges quickly** to the best prompt with natural exploration-exploitation balance.
3. **Rubric-based and CoT prompts** tend to produce more calibrated scores (closer to human), while **strict critics** often under-score.
4. **Agreement varies by quality tier** — judges tend to agree more with humans on high-quality responses and diverge more on low-quality ones.
5. **Cross-provider stability** can be checked — a good judge prompt should work well across OpenAI and Anthropic.
6. **Online learning** allows continuous improvement as human feedback becomes available.

### Next Steps

- **Contextual bandits**: Use query features (type, complexity) to select judge prompts per-context instead of globally.
- **Pairwise evaluation**: Extend to pairwise comparison judge prompts ("Which response is better?").
- **RLHF alignment**: Train a reward model from pairwise human comparisons and use it to fine-tune judge prompts.
- **Multi-step evaluation**: Use Q-Learning to orchestrate multi-step evaluation pipelines (retrieval audit → reasoning check → safety filter).

In [ ]:
print("\n🎉 LLM-as-Judge Prompt Selection via Multi-Armed Bandits — Complete!")
print("\nThis notebook demonstrated:")
print("  • 6 different LLM judge prompt templates as bandit arms")
print("  • 11 evaluation samples with human reference scores (high/med/low quality)")
print("  • 3 bandit algorithms: Thompson Sampling, UCB, Epsilon-Greedy")
print("  • LangGraph state machine for the bandit training loop")
print("  • Agreement metrics: MAE-based + Spearman rank correlation")
print("  • Cross-provider evaluation (OpenAI vs Anthropic)")
print("  • Production-ready AdaptiveJudgeSelector with online learning")
print(f"\n💰 Total session cost: ${llm_service.total_cost:.4f}")